<a href="https://colab.research.google.com/github/Mo1Halem/DEPI_ML/blob/main/Practical_News_Topic_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task: News Topic Classification with AG News

## Objective
Classify **news articles** into 4 categories (*World, Sports, Business, Sci/Tech*) using different **text representation methods**.

<small>[Link: AG News Classification Dataset on Kaggle](https://www.kaggle.com/datasets/amananandrai/ag-news-classification-dataset)</small>
    
---

## Step 1: Data Preparation
- Load the **AG News dataset** (train.csv & test.csv).  
- Combine the **title + description** into one text field.  
- Apply **basic preprocessing** (depend of the task)
---

## Step 2: Representations to Try
You must implement **all 5 methods** below:

1. **One hot Encoding**  (NOW)

2. **Bag of Words (BoW)**   (NOW)
   - Represent each text as a count of words.  


3. **TF-IDF**   (NOW)
   - Apply TF-IDF weighting instead of raw counts.  


4. **N-grams (Bi/Tri-grams)**   (NOW)
   - Use bigrams and trigrams to capture context.   

5. Word2Vec

## Step 3: Try Two Classifiers
For **each text representation method**, train **two different models** and compare:

- **Logistic Regression**
- **Naive Bayes** (or any other model of your choice, e.g., SVM, Decision Tree)

Hint:  
- Logistic Regression usually performs well on sparse features (BoW, TF-IDF, N-grams).  
- Naive Bayes is very fast and works surprisingly well for text classification.  
- Compare their accuracy for each representation.

---

## Step 4: Ali Results Table
Fill in your results:

| Representation | Logistic Regression Acc | Naive Bayes Acc | Notes |
|----------------|--------------------------|-----------------|-------|
| One Hot            |                          |                 |       |
| BoW            |                          |                 |       |
| TF-IDF         |                          |                 |       |
| N-grams        |                          |                 |       |
| Word2Vec        |                          |                 |       |

---

## Reflection Questions
1. Which method gave the best accuracy? Why?  
2. Which method is more efficient in terms of speed and memory?  
3. If you had to build a **real news classifier**, which method would you choose and why?  


In [4]:
pip install datasets scikit-learn gensim nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 37.3 MB/s  0:00:00


In [5]:
import re, nltk, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from sklearn.utils import shuffle
from gensim.models import Word2Vec

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("amananandrai/ag-news-classification-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'ag-news-classification-dataset' dataset.
Path to dataset files: /kaggle/input/ag-news-classification-dataset


In [6]:
train = pd.read_csv(f"{path}/train.csv")
test = pd.read_csv(f"{path}/test.csv")

for df in (train, test):
    df["text"] = df["Title"] + " " + df["Description"]
    df["label"] = df["Class Index"] - 1

print(train.shape, test.shape)
train[["text", "label"]].head()

(120000, 5) (7600, 5)


,text,label
0,Wall St. Bears Claw Back Into the Black (Reute...,2
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2
3,Iraq Halts Oil Exports from Main Southern Pipe...,2
4,"Oil prices soar to all-time record, posing new...",2


In [10]:
train.head()

,Class Index,Title,Description,text,label
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli...",Wall St. Bears Claw Back Into the Black (Reute...,2
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...,Carlyle Looks Toward Commercial Aerospace (Reu...,2
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...,Oil and Economy Cloud Stocks' Outlook (Reuters...,2
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...,Iraq Halts Oil Exports from Main Southern Pipe...,2
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco...","Oil prices soar to all-time record, posing new...",2


In [9]:
train_s = train.sample(5000, random_state=42)
test_s = test.sample(1000, random_state=42)

# Preprocessing

In [12]:
nltk.download("stopwords")
stop_words = set(stopwords.words("english"))

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = [t for t in text.split() if t not in stop_words]
    return " ".join(tokens)

train_texts = [preprocess(t) for t in train_s["text"]]
test_texts = [preprocess(t) for t in test_s["text"]]

train_labels = train_s["label"].tolist()
test_labels = test_s["label"].tolist()

train_texts[0]

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


'bbc set major shake claims newspaper london british broadcasting corporation world biggest public broadcaster cut almost quarter strong workforce biggest shake year history times newspaper london said monday'

# 1) One-Hot

In [13]:
onehot = CountVectorizer(binary=True)
X_train_oh = onehot.fit_transform(train_texts)
X_test_oh = onehot.transform(test_texts)

print(X_train_oh.shape)

(5000, 17186)


In [14]:
def evaluate(X_train, X_test):
    lr = LogisticRegression(max_iter=1000).fit(X_train, train_labels)
    nb = MultinomialNB().fit(X_train, train_labels)
    return (accuracy_score(test_labels, lr.predict(X_test)),
            accuracy_score(test_labels, nb.predict(X_test)))

results = {}
results["One Hot"] = evaluate(X_train_oh, X_test_oh)
results

{'One Hot': (0.864, 0.872)}

# 2) BOW

In [15]:
bow = CountVectorizer()
X_train_bow = bow.fit_transform(train_texts)
X_test_bow = bow.transform(test_texts)

results["BoW"] = evaluate(X_train_bow, X_test_bow)
results

{'One Hot': (0.864, 0.872), 'BoW': (0.862, 0.882)}

# 3) TF-IDF

In [16]:
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(train_texts)
X_test_tfidf = tfidf.transform(test_texts)

results["TF-IDF"] = evaluate(X_train_tfidf, X_test_tfidf)
results

{'One Hot': (0.864, 0.872), 'BoW': (0.862, 0.882), 'TF-IDF': (0.873, 0.871)}

# 4) N-grams

In [20]:
ngram = CountVectorizer(ngram_range=(1, 3))     # لما استخدمت (2, 3) Bi/Tri-grams بس طلعت :  'N-grams': (0.76, 0.819)
X_train_ng = ngram.fit_transform(train_texts)
X_test_ng = ngram.transform(test_texts)

print(X_train_ng.shape)
results["N-grams"] = evaluate(X_train_ng, X_test_ng)
results

(5000, 226232)


{'One Hot': (0.864, 0.872),
 'BoW': (0.862, 0.882),
 'TF-IDF': (0.873, 0.871),
 'N-grams': (0.859, 0.877)}

# 5) Word2Vec

In [21]:
# عشان MultinomialNB بيرفض القيم السالبة
from sklearn.naive_bayes import GaussianNB

def evaluate(X_train, X_test, nb_class=MultinomialNB):
    lr = LogisticRegression(max_iter=1000).fit(X_train, train_labels)
    nb = nb_class().fit(X_train, train_labels)
    return (accuracy_score(test_labels, lr.predict(X_test)),
            accuracy_score(test_labels, nb.predict(X_test)))

In [22]:
train_tokens = [text.split() for text in train_texts]

w2v = Word2Vec(sentences=train_tokens, vector_size=100,
               window=5, min_count=1, sg=1, epochs=10)

In [23]:
def get_vector(text):
    vectors = [w2v.wv[w] for w in text.split() if w in w2v.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(100)

X_train_w2v = np.array([get_vector(t) for t in train_texts])
X_test_w2v = np.array([get_vector(t) for t in test_texts])

print(X_train_w2v.shape, X_test_w2v.shape)

(5000, 100) (1000, 100)


In [24]:
results["Word2Vec"] = evaluate(X_train_w2v, X_test_w2v, nb_class=GaussianNB)
results

{'One Hot': (0.864, 0.872),
 'BoW': (0.862, 0.882),
 'TF-IDF': (0.873, 0.871),
 'N-grams': (0.859, 0.877),
 'Word2Vec': (0.84, 0.795)}

Results

In [25]:
df_results = pd.DataFrame(results, index=["Logistic Regression", "Naive Bayes"]).T
df_results = (df_results * 100).round(1)
df_results

,Logistic Regression,Naive Bayes
One Hot,86.4,87.2
BoW,86.2,88.2
TF-IDF,87.3,87.1
N-grams,85.9,87.7
Word2Vec,84.0,79.5
